KAGGLE DA YAPILAN İŞLEMLER ÇALIŞTIRILMADAN AÇIKLAMASI İLE YAZILMIŞTIR

In [ ]:
""" VERİ YÜKLENDİ """
import polars as pl

DATA_PATH = "/kaggle/input/datasets/denizbyat/labeled/metropt3_labeled.parquet"

df = pl.read_parquet(DATA_PATH)

print(f"Satır sayısı : {df.shape[0]:,}")
print(f"Sütun sayısı : {df.shape[1]}")
print(f"Sütunlar     : {df.columns}")

In [ ]:
""" TEMİZLİK """
df = df.drop(['time_diff', 'session_id'])

# Sensör sütunları
sensor_cols = [
    'TP2', 'TP3', 'H1', 'DV_pressure', 'Reservoirs',
    'Oil_temperature', 'Motor_current', 'COMP', 'DV_eletric',
    'Towers', 'MPG', 'LPS', 'Pressure_switch', 'Oil_level',
    'Caudal_impulses'
]

print(f"Temizlendi — kalan sütunlar: {df.columns}")
print(f"RAM kullanımı: {df.estimated_size('mb'):.1f} MB")

time_diff ve session_id model için gereksizdi onları temizledik

In [ ]:
""" PENCERELER VE İSTATİSTİK (Rolling Features) """
# pencerelerin boyutları
windows = { 
    '1h':  360,
    '6h':  2160,
    '24h': 8640
}

# Rolling features üret
rolling_features = [df]

for window_name, window_size in windows.items():    # pencereleri sensörlerin üzerinde gezdiriyoruz
    for col in sensor_cols:
        rolling_features.append(    # veris setine bilgileri ekliyoruz
            df.select([
                pl.col(col)
                  .rolling_mean(window_size=window_size, min_samples=1) # ortalama değerleri
                  .alias(f'{col}_rmean_{window_name}'),
                pl.col(col)
                  .rolling_std(window_size=window_size, min_samples=1) # std değerleri
                  .alias(f'{col}_rstd_{window_name}')
            ])
        )

# Hepsini birleştir
df_features = pl.concat(rolling_features, how='horizontal')

print(f"Toplam özellik sayısı : {df_features.shape[1]}")
print(f"RAM kullanımı         : {df_features.estimated_size('mb'):.1f} MB")

3 adet pencere oluşturuyoruz. Her sensöre şuan ki değer ile oluşturduğumuz pencerelerin zaman aralığında ki değerlerle normal mi. Her pencere için ortalama ve standart sapma hesaplatarak bunun değerlendirmesini yapıyoruz. Toplamda : 15 sensör × 3 pencere × 2 istatistik = 90 yeni özellik + 17 tanede var saten 107 özellik olacak

In [ ]:
""" KISA PENCERELER (Lag Features) """
lag_steps = {
    '10min': 60,   # 60 kayıt = 10 dakika
    '1h':    360,  # 360 kayıt = 1 saat
    '6h':    2160  # 2160 kayıt = 6 saat
}

lag_features = []

for lag_name, lag_size in lag_steps.items():    # lag adları ve boyutları üzerinde geziniyoruz
    for col in sensor_cols:
        lag_features.append(
            df_features.select([
                # Geçmiş değer
                pl.col(col)
                  .shift(lag_size)
                  .alias(f'{col}_lag_{lag_name}'),
                # Şimdiki - geçmiş = değişim miktarı (anlık değişim - rollingden en büyük fark)
                (pl.col(col) - pl.col(col).shift(lag_size))
                  .alias(f'{col}_diff_{lag_name}')
            ])
        )

df_features = pl.concat([df_features] + lag_features, how='horizontal')     # tüm özellikleri birleştir

print(f"Toplam özellik sayısı : {df_features.shape[1]}")
print(f"RAM kullanımı         : {df_features.estimated_size('mb'):.1f} MB")

Önceki adımda bulunan pencerelerin daha küçük ancak anlık duruma bakan halleri olarak düşünülebilir. Rolling de her pencerenin içinde kalan değerleri düzleştirerek (ortalama ve stand. sap. alarak) gürültüyü azaltıyorduk. Elimizde her pencere için tek değer kalıyor ve trend takibi yapıyorduk. Burada ise denk gelen her değeri o anki durum ile karşılaştırması yapılıyor yani ani değişimler nokta atış tespit edilerek anormallik aranıyor.

Toplamda 197 adet özellik elde edildi.

In [ ]:
""" ZAMAN VE SENSÖRLERİN BİRLEŞTİRİLMESİ ÖZELLİKLERİ (Çaprazlama ve Zamanlama) """
# Zaman özellikleri
time_features = df_features.select([
    pl.col('timestamp').dt.hour().alias('hour_of_day'),     # günün saatini çıkarıyoruz
    pl.col('timestamp').dt.weekday().alias('day_of_week'),  # haftanın günleri
    pl.col('timestamp').dt.month().alias('month'),      # ay bilgisi (mevsimsellik için) 
])

# Çapraz sensör özellikleri (sensörleri birleştirerek yeni özellikler oluşturuyoruz)
cross_features = df_features.select([
    # Basınç oranı — TP2/TP3 oranı normalden saptığında anomali sinyali oluşturabilir (hava kaçağı gibi)
    (pl.col('TP2') / (pl.col('TP3') + 1e-6))
        .alias('TP2_TP3_ratio'),
    
    # Motor yük endeksi — akım × sıcaklık (çift stres yaratır yüksek baskın altında)
    (pl.col('Motor_current') * pl.col('Oil_temperature'))
        .alias('motor_heat_index'),
    
    # Basınç farkı — TP3 ile H1 arasındaki fark (sızıntı veya tıkanıklık göstergesi olabilir)
    (pl.col('TP3') - pl.col('H1'))
        .alias('pressure_diff_TP3_H1'),
    
    # DV_pressure × Motor_current — basınç altında motor yükü (kompresörün zorlandığı durumlarda artar)
    (pl.col('DV_pressure') * pl.col('Motor_current'))
        .alias('pressure_motor_load'),
])

# Hepsini birleştir
df_features = pl.concat([df_features, time_features, cross_features], how='horizontal')

print(f"Toplam özellik sayısı : {df_features.shape[1]}")
print(f"RAM kullanımı         : {df_features.estimated_size('mb'):.1f} MB")

Farklı sensörleri birleştirerek sistem hakkında daha detaylı bilgiler edinebileceğimiz 3 zaman - 4 çapraz sensör verisi olmak üzere toplamda 7 özellik daha ekledik ve toplamda 204 özelliğimiz oldu

In [ ]:
""" FREKANS (TİTREŞİM) ÖZELLİĞİ - FFT """
import numpy as np

# FFT — Motor_current için son 1 saatlik pencere
window_size = 360  # 1 saat = 360 kayıt

motor_values = df_features['Motor_current'].to_numpy()  # Motor_current değerlerini numpy dizisine çeviriyoruz (matematiksel işlemler için numpy gerekli)

# her özellik için (baskın frekans, baskın güç ve toplam güç) sıfır dizileri oluşturuyoruz (ayrı ayrı hesaplama yapılacak)
fft_dominant_freq = np.zeros(len(motor_values))
fft_dominant_power = np.zeros(len(motor_values))
fft_total_power = np.zeros(len(motor_values))

for i in range(window_size, len(motor_values)):     # belirlediğimiz zaman penceresi boyunca tek tek dizilerde geziniyoruz
    window = motor_values[i - window_size:i]        # mevcut pencereyi alıyoruz (son 1 saatlik motor akım değerleri)
    
    # FFT hesapla
    fft_vals = np.fft.rfft(window)      # Gerçek değerler için hızlı Fourier dönüşümü (FFT) hesaplıyoruz yani burada değerlerin gerçek ve sanal cinsinden (3.2 + 4.1j gibi) frekans bileşenlerini buluyoruz
    # toplamda (n / 2) + 1 formülünden (360 / 2) + 1 = 181 adet frekans bileşeni elde ederiz 
    # Bunun sebebi FFT nin simetrik olmasıdır. aslında her değer için bir pozitif ve negatif frekans vardır ama gerçek değerler için negatif frekanslar aynıdır yani 1.değer frekansı = son değer yani 360.değer frekansına bu yüzden sadece pozitifleri alırız bu da yarısıdır.

    # Güç hesaplama
    fft_power = np.abs(fft_vals) ** 2   # Gücü hesaplamak için .abs ile yukarıda ki frekansların önce gerçek ve sanal kısımların sayısal kısımlarının (gerçek saten sayısaldır sanak kısım için ise j ifadesi olamadan) karesinin karekökünü alıyoruz yani karmaşık sayının büyüklüğünü buluyoruz
    # sonrasında bu büyüklüğün karesini alarak gücü hesaplıyoruz.
    # FFT'nin her bileşeninin gücü, o frekanstaki titreşim enerjisini temsil eder. Yani yüksek güç, o frekansta güçlü bir titreşim olduğunu gösterir.
    
    # DC bileşenini çıkar (index 0 — sabit ortalama, bize lazım değil çünkü dc bileşeni sistemin ortalamasıdır, titreşimle ilgili bilgi vermez bize ac yani değişen bileşenler lazım)
    fft_power[0] = 0 
    
    # Baskın frekans ve gücü al
    dominant_idx = np.argmax(fft_power) # Gücü en yüksek olan frekansın indeksini buluyoruz yani en güçlü titreşim hangi frekansta var onu buluyoruz    
    fft_dominant_freq[i] = dominant_idx # Baskın frekansın indeksini frekans değerine dönüştürmek için (örneğin, 1. indeks = 1/360 Hz, 2. indeks = 2/360 Hz, vb.)
    fft_dominant_power[i] = fft_power[dominant_idx] # Baskın frekansın gücünü kaydediyoruz
    fft_total_power[i] = np.sum(fft_power)  # Toplam gücü kaydediyoruz (tüm frekansların gücünün toplamı, titreşim enerjisinin genel bir göstergesidir)

# Yeni özellikleri DataFrame'e ekle
fft_df = pl.DataFrame({
    'fft_dominant_freq': fft_dominant_freq.astype(np.float32),
    'fft_dominant_power': fft_dominant_power.astype(np.float32),
    'fft_total_power': fft_total_power.astype(np.float32)
})

df_features = pl.concat([df_features, fft_df], how='horizontal')

print(f"Toplam özellik sayısı : {df_features.shape[1]}")
print(f"RAM kullanımı         : {df_features.estimated_size('mb'):.1f} MB")

Sistemde doğrudan titreşim sensörümüz bulunmadığından bu frekans özelliğinden yüksek sonuç beklemiyoruz ancak titreşim sensörüne benzer (belli frekansta değer üreten) çıktılar üreten ve dolaylı yoldan titreşimle bağlantılı olan birkaç sensörümüz var. Bunlar motor akımı (motor_current) sensörü, yağ sıcaklığı ve TP2,3 basınç döngüleri olabilir. Yağ ve basınç döngüleri net bir frekansda olduklarından ve değişimleri çok sınırlı olduklarından bu sensörlerde arama yapmak gereksiz yorar ancak motor akım sensöründe yüksek ihtimalle fayda sağlayabiliriz. Fayda olmaması durumunda zaten özellik seçim adımında elenecektir.

Burada önce sensörün belirlenen zaman aralığında ki değerlerini alarak o dönemdeki frekansını (normal-en çok tekrar eden değer) bulup diğer zamanlar ile karşılaştırarak frekans dışı değerlerde anomali tespiti yapıyor. Sonucunda 3 yeni özellik oluşturuluyor: dominant_freq, döneme ait en güçlü frekans - dominant_power, frekansın gücü - total_power, frekansların toplam gücü

In [ ]:
""" EKSİK DEĞERLER VE TEMİZLEME """
null_counts = df_features.null_count()
null_cols = [col for col in df_features.columns 
             if null_counts[col][0] > 0]

print(f"Eksik değer içeren sütun sayısı: {len(null_cols)}")

# En uzun pencere 24 saat = 8640 kayıt o kadar satırı at
df_features = df_features.slice(8640)
print(f"Satır sayısı (temizleme sonrası): {df_features.shape[0]:,}")

Eksik değerler lag ve rolling özellikleri başında ki boşluklar temizlendi. Rollingde en yüksek pencereyi 24 saat yapmıştık yani 8640 kayıt ondan dolayı slice içinede bu miktarı yazdıl.

In [ ]:
""" 0 VARYANSLI TEMİZLİK """
numeric_cols = [        # sayı içeren sütunları seçiyoruz (
    col for col in df_features.columns
    if col not in ['timestamp', 'is_suspect']   # timestamp ve is_suspect hariç, çünkü onlar sayısal değil
    and df_features[col].dtype != pl.Utf8    # string türündeki sütunları da hariç tutuyoruz
    and df_features[col].dtype != pl.Datetime   # datetime türündeki sütunları da hariç tutuyoruz
]

zero_var_cols = [       # standart sapması sıfır olan sütunları buluyoruz yani tüm değerleri aynı olan sütunlar (gereksiz ağırlık yapar)
    col for col in numeric_cols
    if df_features[col].std() == 0
]

print(f"Sayısal sütun sayısı         : {len(numeric_cols)}")
print(f"Sıfır varyanslı sütun sayısı : {len(zero_var_cols)}")

if zero_var_cols:       # sıfır varyanslı sütunlar varsa atıyoruz
    print(f"Atılan sütunlar: {zero_var_cols}")
    df_features = df_features.drop(zero_var_cols)
else:
    print("Atılacak sütun yok")

205 sayısal sütunun tamamı anlamlı değer içeriyor sıfır varyanslı sütun çıkmadı, herhangi bir sütun atılmadı.

In [ ]:
""" KORELASYON HESAPLAMA (TÜM ÖZELLİKLER İÇİN) """
import numpy as np

# korelasyon matrisini hesaplıyoruz (sadece sayısal sütunlar üzerinde -- .abs ile mutlak değer alarak pozitif korelasyonları da dikkate alıyoruz)
corr_matrix = df_features.select(numeric_cols).to_pandas().corr().abs()

# Üst üçgeni al (çünkü korelasyon matrisi simetriktir ve kendisiyle korelasyonu 1 olduğu için diagonal de gereksizdir)
upper = corr_matrix.where(      # .where ile dikkate alınacak kısmı gösteriyoruz
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)   
    # triu bizim matris boyutu kadar (yani 205 özellik var 205x205 boyutunda) bir matris oluşturup sıfırlardan üst üçgenini 1 yapar yani sadece üst üçgeni alırız (k=1 ile diagonal de sıfırlanır yani kendisiyle korelasyon dikkate alınmaz)
)

# %95 üzeri korelasyonlu sütunları bul
high_corr_cols = [
    col for col in upper.columns    # korelasyon matrisinin sütunları üzerinde geziniyoruz
    if any(upper[col] > 0.95)   # herhangi bir sütunla %95'ten yüksek korelasyona sahip olan sütunları seçiyoruz
]

print(f"%95+ korelasyonlu sütun sayısı : {len(high_corr_cols)}")
print(f"Atılan sütunlar (ilk 10)       : {high_corr_cols[:10]}")

df_features = df_features.drop(high_corr_cols)      # yüksek korelasyonlu sütunları atıyoruz

print(f"\nFinal özellik sayısı : {df_features.shape[1]}")
print(f"RAM kullanımı        : {df_features.estimated_size('mb'):.1f} MB")

çıktıda 207 özelliğimizden 74 tanesi birbiri ile korelasyonlu çıktı (kendi aralarında 2-3 tanesi birbirileriyle şeklinde) bunları attıktan sonra geriye 133 özelliğimiz kaldı. 95 üzeri korelasyon birbiri ile tıpa tıp hareket gösteren özelliklerdir. bunlar hem gereksiz ağırlık hem de model yanıltması yapar.